# Лабораторна робота №3. Робота з NoSQL (MongoDB): імпорт, експорт, фільтрація даних

**Мета роботи:** ознайомитися з MongoDB: створення БД і колекцій, імпорт/експорт JSON і CSV, фільтрація, сортування, агрегація.

**Коротко про теорію.** MongoDB — документно-орієнтована NoSQL-СУБД.
- **База даних** (`university`) — контейнер колекцій.
- **Колекція** (`students`) — набір документів, аналог таблиці в SQL, але без жорсткої схеми.
- **Документ** — JSON-подібний (BSON) об'єкт, аналог рядка таблиці.
- **Поле** (`name`, `age`, `major`, `gpa`) — пара «ключ — значення» в документі, аналог стовпця. Кожен документ має унікальне поле `_id`.

Передумова: сервер `mongod` запущено локально на порту 27017.

## Підготовка: версії та підключення

In [1]:
import os

# Щоб `!`-команди знаходили mongosh / mongoimport / mongoexport (Homebrew)
if "/opt/homebrew/bin" not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = "/opt/homebrew/bin" + os.pathsep + os.environ["PATH"]

import pandas as pd
import pymongo
from pymongo import MongoClient

print("pymongo:", pymongo.version)

pymongo: 4.20.1


In [2]:
!mongod --version | head -1
!mongosh --version
!mongoimport --version | head -1

db version v9.0.8


2.14.2


mongoimport version: 100.16.1


In [3]:
client = MongoClient("mongodb://localhost:27017/")
print(client.admin.command("ping"))

# Для перезапуску ноутбука починаємо з чистої бази
client.drop_database("university")
print("Бази:", client.list_database_names())

{'ok': 1.0}
Бази: ['admin', 'config', 'local']


## Крок 2. Створення бази даних і колекції

База `university` і колекція `students` створюються автоматично під час першої вставки документів.

In [4]:
!mongosh university --quiet --eval 'db.students.insertMany([{name: "Ivan Petrenko", age: 20, major: "Computer Science", gpa: 3.8}, {name: "Olena Ivanenko", age: 22, major: "Information Systems", gpa: 3.5}, {name: "Petro Kovalchuk", age: 21, major: "Software Engineering", gpa: 3.9}]).insertedIds.length' | cat

In [5]:
!mongosh university --quiet --eval 'db.students.countDocuments()' | cat

3


## Крок 3. Імпорт і експорт даних

Файл `students.json` містить 10 студентів (перші троє — ті самі, що вже вставлені). Тому використовуємо `--drop`: він видаляє колекцію перед імпортом, інакше перші троє студентів задублювалися б. Це також виконує п. 1 самостійного завдання.

In [6]:
!mongoimport --db university --collection students --file students.json --jsonArray --drop

2026-11-20T15:14:14.761+0200	connected to: mongodb://localhost/
2026-11-20T15:14:14.761+0200	dropping: `university.students`
2026-11-20T15:14:14.792+0200	10 document(s) imported successfully. 0 document(s) failed to import.


In [7]:
!mongosh university --quiet --eval 'db.students.countDocuments()' | cat

10


In [8]:
!mongoexport --db university --collection students --type=csv --fields name,age,major,gpa --out students.csv
pd.read_csv("students.csv")

2026-11-20T15:14:15.888+0200	connected to: mongodb://localhost/
2026-11-20T15:14:15.889+0200	exported 10 records


,name,age,major,gpa
0,Ivan Petrenko,20,Computer Science,3.8
1,Olena Ivanenko,22,Information Systems,3.5
2,Oksana Melnyk,24,Information Systems,3.2
3,Maria Bondarenko,19,Computer Science,3.4
4,Petro Kovalchuk,21,Software Engineering,3.9
5,Andrii Shevchenko,23,Data Science,3.7
6,Iryna Moroz,20,Data Science,3.9
7,Bohdan Lysenko,22,Computer Science,3.0
8,Dmytro Tkachenko,21,Software Engineering,3.6
9,Sofia Kravchenko,18,Data Science,4.0


## Крок 4. Базові запити

Кожен запит виконано двічі: у `mongosh` (щоб показати синтаксис shell) і через `pymongo` з виводом у DataFrame.

In [9]:
db = client["university"]
students = db["students"]

**4.1. Усі документи**

In [10]:
!mongosh university --quiet --eval 'printjson(db.students.find({}, {_id: 0}).toArray())' | cat

[
  {
    name: 'Ivan Petrenko',
    age: 20,
    major: 'Computer Science',
    gpa: 3.8
  },
  {
    name: 'Olena Ivanenko',
    age: 22,
    major: 'Information Systems',
    gpa: 3.5
  },
  {
    name: 'Oksana Melnyk',
    age: 24,
    major: 'Information Systems',
    gpa: 3.2
  },
  {
    name: 'Maria Bondarenko',
    age: 19,
    major: 'Computer Science',
    gpa: 3.4
  },
  {
    name: 'Petro Kovalchuk',
    age: 21,
    major: 'Software Engineering',
    gpa: 3.9
  },
  {
    name: 'Andrii Shevchenko',
    age: 23,
    major: 'Data Science',
    gpa: 3.7
  },
  {
    name: 'Iryna Moroz',
    age: 20,
    major: 'Data Science',
    gpa: 3.9
  },
  {
    name: 'Bohdan Lysenko',
    age: 22,
    major: 'Computer Science',
    gpa: 3
  },
  {
    name: 'Dmytro Tkachenko',
    age: 21,
    major: 'Software Engineering',
    gpa: 3.6
  },
  {
    name: 'Sofia Kravchenko',
    age: 18,
    major: 'Data Science',
    gpa: 4
  }
]


In [11]:
pd.DataFrame(list(students.find({}, {"_id": 0})))

,name,age,major,gpa
0,Ivan Petrenko,20,Computer Science,3.8
1,Olena Ivanenko,22,Information Systems,3.5
2,Oksana Melnyk,24,Information Systems,3.2
3,Maria Bondarenko,19,Computer Science,3.4
4,Petro Kovalchuk,21,Software Engineering,3.9
5,Andrii Shevchenko,23,Data Science,3.7
6,Iryna Moroz,20,Data Science,3.9
7,Bohdan Lysenko,22,Computer Science,3.0
8,Dmytro Tkachenko,21,Software Engineering,3.6
9,Sofia Kravchenko,18,Data Science,4.0


**4.2. Фільтр за спеціальністю** `major == "Computer Science"`

In [12]:
!mongosh university --quiet --eval 'printjson(db.students.find({major: "Computer Science"}, {_id: 0}).toArray())' | cat

[
  {
    name: 'Ivan Petrenko',
    age: 20,
    major: 'Computer Science',
    gpa: 3.8
  },
  {
    name: 'Maria Bondarenko',
    age: 19,
    major: 'Computer Science',
    gpa: 3.4
  },
  {
    name: 'Bohdan Lysenko',
    age: 22,
    major: 'Computer Science',
    gpa: 3
  }
]


In [13]:
pd.DataFrame(list(students.find({"major": "Computer Science"}, {"_id": 0})))

,name,age,major,gpa
0,Ivan Petrenko,20,Computer Science,3.8
1,Maria Bondarenko,19,Computer Science,3.4
2,Bohdan Lysenko,22,Computer Science,3.0


**4.3. Фільтр `gpa > 3.6`**

In [14]:
!mongosh university --quiet --eval 'printjson(db.students.find({gpa: {$gt: 3.6}}, {_id: 0}).toArray())' | cat

[
  {
    name: 'Ivan Petrenko',
    age: 20,
    major: 'Computer Science',
    gpa: 3.8
  },
  {
    name: 'Petro Kovalchuk',
    age: 21,
    major: 'Software Engineering',
    gpa: 3.9
  },
  {
    name: 'Andrii Shevchenko',
    age: 23,
    major: 'Data Science',
    gpa: 3.7
  },
  {
    name: 'Iryna Moroz',
    age: 20,
    major: 'Data Science',
    gpa: 3.9
  },
  {
    name: 'Sofia Kravchenko',
    age: 18,
    major: 'Data Science',
    gpa: 4
  }
]


In [15]:
pd.DataFrame(list(students.find({"gpa": {"$gt": 3.6}}, {"_id": 0})))

,name,age,major,gpa
0,Ivan Petrenko,20,Computer Science,3.8
1,Petro Kovalchuk,21,Software Engineering,3.9
2,Andrii Shevchenko,23,Data Science,3.7
3,Iryna Moroz,20,Data Science,3.9
4,Sofia Kravchenko,18,Data Science,4.0


**4.4. Сортування за `gpa` за спаданням**

In [16]:
!mongosh university --quiet --eval 'printjson(db.students.find({}, {_id: 0}).sort({gpa: -1}).toArray())' | cat

[
  {
    name: 'Sofia Kravchenko',
    age: 18,
    major: 'Data Science',
    gpa: 4
  },
  {
    name: 'Petro Kovalchuk',
    age: 21,
    major: 'Software Engineering',
    gpa: 3.9
  },
  {
    name: 'Iryna Moroz',
    age: 20,
    major: 'Data Science',
    gpa: 3.9
  },
  {
    name: 'Ivan Petrenko',
    age: 20,
    major: 'Computer Science',
    gpa: 3.8
  },
  {
    name: 'Andrii Shevchenko',
    age: 23,
    major: 'Data Science',
    gpa: 3.7
  },
  {
    name: 'Dmytro Tkachenko',
    age: 21,
    major: 'Software Engineering',
    gpa: 3.6
  },
  {
    name: 'Olena Ivanenko',
    age: 22,
    major: 'Information Systems',
    gpa: 3.5
  },
  {
    name: 'Maria Bondarenko',
    age: 19,
    major: 'Computer Science',
    gpa: 3.4
  },
  {
    name: 'Oksana Melnyk',
    age: 24,
    major: 'Information Systems',
    gpa: 3.2
  },
  {
    name: 'Bohdan Lysenko',
    age: 22,
    major: 'Computer Science',
    gpa: 3
  }
]


In [17]:
pd.DataFrame(list(students.find({}, {"_id": 0}).sort("gpa", -1)))

,name,age,major,gpa
0,Sofia Kravchenko,18,Data Science,4.0
1,Petro Kovalchuk,21,Software Engineering,3.9
2,Iryna Moroz,20,Data Science,3.9
3,Ivan Petrenko,20,Computer Science,3.8
4,Andrii Shevchenko,23,Data Science,3.7
5,Dmytro Tkachenko,21,Software Engineering,3.6
6,Olena Ivanenko,22,Information Systems,3.5
7,Maria Bondarenko,19,Computer Science,3.4
8,Oksana Melnyk,24,Information Systems,3.2
9,Bohdan Lysenko,22,Computer Science,3.0


**4.5. Кількість документів**

In [18]:
!mongosh university --quiet --eval 'db.students.countDocuments()' | cat
print(students.count_documents({}))

10


10


## Крок 5. Робота з MongoDB з Python

In [19]:
from pymongo import MongoClient
import pandas as pd

client = MongoClient("mongodb://localhost:27017/")
db = client["university"]
collection = db["students"]

data = list(collection.find({}, {"_id": 0}))
df = pd.DataFrame(data)
print(df)

print(df[df["gpa"] > 3.7])

                name  age                 major  gpa
0      Ivan Petrenko   20      Computer Science  3.8
1     Olena Ivanenko   22   Information Systems  3.5
2      Oksana Melnyk   24   Information Systems  3.2
3   Maria Bondarenko   19      Computer Science  3.4
4    Petro Kovalchuk   21  Software Engineering  3.9
5  Andrii Shevchenko   23          Data Science  3.7
6        Iryna Moroz   20          Data Science  3.9
7     Bohdan Lysenko   22      Computer Science  3.0
8   Dmytro Tkachenko   21  Software Engineering  3.6
9   Sofia Kravchenko   18          Data Science  4.0
               name  age                 major  gpa
0     Ivan Petrenko   20      Computer Science  3.8
4   Petro Kovalchuk   21  Software Engineering  3.9
6       Iryna Moroz   20          Data Science  3.9
9  Sofia Kravchenko   18          Data Science  4.0


## Самостійне завдання

### 1. Імпорт 10 студентів

Виконано в кроці 3 (`mongoimport ... --jsonArray --drop`), у колекції 10 документів.

### 2. Фільтрація за віком і спеціальністю

In [20]:
# Вік >= 21 І спеціальність Computer Science або Software Engineering
query = {"age": {"$gte": 21}, "major": {"$in": ["Computer Science", "Software Engineering"]}}
pd.DataFrame(list(students.find(query, {"_id": 0})))

,name,age,major,gpa
0,Petro Kovalchuk,21,Software Engineering,3.9
1,Bohdan Lysenko,22,Computer Science,3.0
2,Dmytro Tkachenko,21,Software Engineering,3.6


In [21]:
# Приклад із $or: gpa вищий за 3.8 АБО вік менший за 20
query_or = {"$or": [{"gpa": {"$gt": 3.8}}, {"age": {"$lt": 20}}]}
pd.DataFrame(list(students.find(query_or, {"_id": 0})))

,name,age,major,gpa
0,Maria Bondarenko,19,Computer Science,3.4
1,Petro Kovalchuk,21,Software Engineering,3.9
2,Iryna Moroz,20,Data Science,3.9
3,Sofia Kravchenko,18,Data Science,4.0


### 3–4. Середній GPA за спеціальностями та збереження в колекцію `results`

Конвеєр агрегації групує студентів за спеціальністю, рахує середній GPA та кількість, а стадія `$out` записує результат у колекцію `results`.

In [22]:
pipeline = [
    {"$group": {"_id": "$major", "avg_gpa": {"$avg": "$gpa"}, "students_count": {"$sum": 1}}},
    {"$project": {"_id": 0, "major": "$_id", "avg_gpa": {"$round": ["$avg_gpa", 2]}, "students_count": 1}},
    {"$sort": {"avg_gpa": -1}},
    {"$out": "results"},
]
students.aggregate(pipeline)

results = db["results"]
print("Документів у results:", results.count_documents({}))
res_df = pd.DataFrame(list(results.find({}, {"_id": 0}))).sort_values("avg_gpa", ascending=False).reset_index(drop=True)
res_df

Документів у results: 4


,students_count,major,avg_gpa
0,3,Data Science,3.87
1,2,Software Engineering,3.75
2,3,Computer Science,3.40
3,2,Information Systems,3.35


Перевірка: той самий розрахунок через `pandas.groupby`.

In [23]:
df = pd.DataFrame(list(students.find({}, {"_id": 0})))
pd_df = (df.groupby("major")
           .agg(avg_gpa=("gpa", "mean"), students_count=("gpa", "size"))
           .round(2).reset_index()
           .sort_values("avg_gpa", ascending=False).reset_index(drop=True))
print(pd_df)

check = res_df.merge(pd_df, on="major", suffixes=("_mongo", "_pandas"))
assert len(check) == len(res_df) == len(pd_df) == 4
assert (check["avg_gpa_mongo"] - check["avg_gpa_pandas"]).abs().max() < 1e-9
assert (check["students_count_mongo"] == check["students_count_pandas"]).all()
print("\nСередні в MongoDB і pandas збігаються.")

                  major  avg_gpa  students_count
0          Data Science     3.87               3
1  Software Engineering     3.75               2
2      Computer Science     3.40               3
3   Information Systems     3.35               2

Середні в MongoDB і pandas збігаються.


### 5. Експорт `results` у CSV

In [24]:
!mongoexport --db university --collection results --type=csv --fields major,avg_gpa,students_count --out results.csv
pd.read_csv("results.csv")

2026-11-20T15:14:20.464+0200	connected to: mongodb://localhost/
2026-11-20T15:14:20.465+0200	exported 4 records


,major,avg_gpa,students_count
0,Data Science,3.87,3
1,Software Engineering,3.75,2
2,Computer Science,3.40,3
3,Information Systems,3.35,2


## Закриття з'єднання

In [25]:
client.close()
print("З'єднання закрито.")

З'єднання закрито.


## Контрольні запитання

**1. У чому полягає основна різниця між MongoDB і SQL-базами даних?**
- **Модель даних.** SQL-бази зберігають дані в таблицях із жорсткою схемою: стовпці й типи задаються заздалегідь. MongoDB — документно-орієнтована база: дані лежать у колекціях документів, і документи однієї колекції можуть мати різний набір полів (гнучка схема).
- **Зв'язки.** У SQL дані нормалізують, розносять по таблицях і поєднують через `JOIN`. У MongoDB пов'язані дані частіше вкладають в один документ: вкладені об'єкти й масиви. Аналог `JOIN` теж є — стадія `$lookup`, але зазвичай без неї можна обійтися.
- **Мова запитів.** У SQL-баз це SQL. У MongoDB запити описують JSON-подібними об'єктами (`find({...})`), а для складної обробки є конвеєр агрегації (`aggregate([...])`).
- **Масштабування.** SQL-бази переважно масштабують вертикально, тобто потужнішим сервером. MongoDB від початку розрахована на горизонтальне масштабування: шардинг і репліка-сети.
- **Транзакції.** У SQL-базах ACID-транзакції є завжди. У MongoDB операція над одним документом атомарна, а багатодокументні транзакції з'явилися з версії 4.0.

**2. Який формат даних використовується у MongoDB?**
Документи зберігаються у форматі **BSON** (Binary JSON). Це двійкове представлення JSON. Порівняно з JSON воно має додаткові типи: `ObjectId`, `Date`, `Decimal128`, `int32`/`int64`, двійкові дані. Користувач бачить і передає документи як JSON-подібні об'єкти. Кожен документ має унікальне поле `_id`, за замовчуванням типу `ObjectId`. Обмін даними з іншими системами йде через JSON або CSV (`mongoimport`/`mongoexport`).

**3. Як виконати фільтрацію документів за кількома умовами?**
- Кілька полів в одному фільтрі автоматично поєднуються через логічне «І»: `db.students.find({ "major": "Computer Science", "age": { $gte: 21 } })`.
- Явні логічні оператори: `$and`, `$or`, `$nor`, `$not`:
  `db.students.find({ $or: [ { "gpa": { $gt: 3.8 } }, { "age": { $lt: 20 } } ] })`.
- Оператори порівняння: `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$nin`. Для одного поля їх можна поєднати: `{ "age": { $gte: 20, $lte: 22 } }`.
- У Python (pymongo) синтаксис такий самий, тільки оператори записуються рядками: `collection.find({"age": {"$gte": 21}, "major": {"$in": ["Computer Science", "Data Science"]}})`.

**4. Як імпортувати та експортувати дані у MongoDB?**
Для цього є утиліти з пакета MongoDB Database Tools:
- імпорт JSON-масиву: `mongoimport --db university --collection students --file students.json --jsonArray`;
- імпорт CSV: `mongoimport --db university --collection students --type=csv --headerline --file students.csv`;
- експорт у CSV, де треба перелічити поля: `mongoexport --db university --collection students --type=csv --fields name,age,major,gpa --out students.csv`;
- експорт у JSON: `mongoexport --db university --collection students --jsonArray --out students.json`.
Те саме можна зробити з Python (`insert_many` після `json.load`/`pd.read_csv`, а `find` → `DataFrame.to_csv`) або через графічний інтерфейс MongoDB Compass. Для повних резервних копій у BSON використовують `mongodump`/`mongorestore`.

**5. Як підключитися до MongoDB з Python і виконати запит?**
Через бібліотеку `pymongo`:
```python
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")   # підключення до сервера
db = client["university"]                            # база даних
collection = db["students"]                          # колекція

for doc in collection.find({"gpa": {"$gt": 3.6}}, {"_id": 0}).sort("gpa", -1):
    print(doc)

print(collection.count_documents({}))                # кількість документів
client.close()
```
`find()` повертає курсор, який можна перебирати або перетворити на `list` і далі на `pandas.DataFrame`. Агрегацію виконує `collection.aggregate([...])`.

## Висновки

У ході роботи створено базу `university` із колекцією `students` у MongoDB, дані імпортовано з JSON (`mongoimport`) та експортовано у CSV (`mongoexport`). Виконано запити з фільтрацією, сортуванням і підрахунком як у `mongosh`, так і через `pymongo` з поданням результатів у `pandas`. Конвеєр агрегації обчислив середній GPA за спеціальностями та записав його в колекцію `results` (стадія `$out`); результат збігається з розрахунком у `pandas` і експортований у `results.csv`.